In [367]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score,confusion_matrix,classification_report
from sklearn.model_selection import cross_val_score
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
import joblib

In [368]:
# load dataset
web = pd.read_csv('website_blocker_dataset.csv')
# data exploration
print(web.head())
print(web.tail())
print(web.index)
print(web.columns)
web.info()
print(web.describe())

                 Url   Labels
0  labaran batsa.com  Blocked
1         python.org  Allowed
2          harka.com  Blocked
3        physics.com  Allowed
4        gidan harka  Blocked
                      Url   Labels
112         magana jarice  Allowed
113  ummi harka cin gindi  Blocked
114            ruwan dadi  Blocked
115         xallah harka   Blocked
116         xxxvideos.com  Blocked
RangeIndex(start=0, stop=117, step=1)
Index(['Url', 'Labels'], dtype='object')
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 117 entries, 0 to 116
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   Url     117 non-null    object
 1   Labels  117 non-null    object
dtypes: object(2)
memory usage: 2.0+ KB
                Url   Labels
count           117      117
unique          116        2
top     lesbian.com  Allowed
freq              2       68


In [369]:
# data cleaning
web.isnull().sum()
web.dropna(inplace=True)
web.duplicated().sum()
web.drop_duplicates(inplace=True)

In [370]:
# data extraction
X = web['Url']
y = web['Labels']

In [371]:
# Test Alogorith performance
# Algorith performance
models = {
    "LogisticRegression": LogisticRegression(),
    "NaiveBayes": MultinomialNB(),
    "SVM": LinearSVC()
}

for name, model in models.items():

    pipe = Pipeline([
        ("tfidf", TfidfVectorizer(ngram_range=(1,2),stop_words='english')),
        ("model", model)
    ])

    scores = cross_val_score(
        pipe,
        X,
        y,
        cv=3
    )

    print(name, scores.mean())

LogisticRegression 0.7593342330184436
NaiveBayes 0.8277103013945118
SVM 0.8535762483130904


In [372]:
# Choose algorith winner and train model with it
pipe = Pipeline([
    ('tfidf',TfidfVectorizer(ngram_range=(1,2),stop_words='english')),
    ('Model',LinearSVC())
])

In [373]:
X = web["Url"]
y = web["Labels"]
#  train test split
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)
# create model and train
pipe.fit(X_train,y_train)
print('Successfully')

Successfully


In [374]:
predictions = pipe.predict(X_test)
print(predictions)

['Blocked' 'Blocked' 'Allowed' 'Allowed' 'Blocked' 'Allowed' 'Blocked'
 'Allowed' 'Allowed' 'Blocked' 'Allowed' 'Allowed' 'Allowed' 'Allowed'
 'Allowed' 'Blocked' 'Blocked' 'Blocked' 'Blocked' 'Allowed' 'Blocked'
 'Allowed' 'Allowed' 'Allowed']


In [375]:
# Accuracy score
score = accuracy_score(y_test,predictions)
print('Accuracy Score',score)
# check underfitting or overfitting
print('Train',pipe.score(X_train,y_train))
print('Test',pipe.score(X_test,y_test))

Accuracy Score 0.875
Train 1.0
Test 0.875


In [376]:
# confusion matrix
cmatrix = confusion_matrix(y_test,predictions)
print('Confusion Matrix',cmatrix)
# classification report
print(classification_report(y_test,predictions))

Confusion Matrix [[13  2]
 [ 1  8]]
              precision    recall  f1-score   support

     Allowed       0.93      0.87      0.90        15
     Blocked       0.80      0.89      0.84         9

    accuracy                           0.88        24
   macro avg       0.86      0.88      0.87        24
weighted avg       0.88      0.88      0.88        24



In [377]:
# cross validation score
cvalidation = cross_val_score(pipe,X,y,cv=3)
print('Cross Validations Score',cvalidation)

Cross Validations Score [0.8974359  0.79487179 0.86842105]


In [378]:
# new predictions
new_urls = [
    "free porn video",
    "jamb portal",
    "python tutorial",
    "casino bonus",
    "openai.com",
    'instagram.com',
    'facebook login',
    'gindi',
    'tsuliya dadi',
    'cin gindin matan hausawa',
    'bbc hausa news',
    'xan can iskanci gindi',
    'matan bariki',
    'lesbian hausawa'
]

results = pipe.predict(new_urls)
for url, result in zip(new_urls, results):
    print(url, "=>", result)

free porn video => Blocked
jamb portal => Allowed
python tutorial => Allowed
casino bonus => Blocked
openai.com => Allowed
instagram.com => Allowed
facebook login => Allowed
gindi => Blocked
tsuliya dadi => Blocked
cin gindin matan hausawa => Blocked
bbc hausa news => Allowed
xan can iskanci gindi => Blocked
matan bariki => Blocked
lesbian hausawa => Blocked


In [379]:
# save model
joblib.dump(pipe, "blocker_brain.pkl")

print("Model saved successfully")

Model saved successfully


In [380]:
# brain test ability
brain = joblib.load("blocker_brain.pkl")

In [382]:
# test 
tests = [
    "free porn video",
    "jamb portal",
    "python tutorial",
    "casino bonus",
    'xvideos.com',
    'labaran cin gindi',
    'batsa novels',
    'labaran duniya',
    'hausa novels',
    'maryam harka',
    'jamila harka',
    'cin gindi dadi',
    'www.computer networking',
    'student login details'
]

print(brain.predict(tests))

['Blocked' 'Allowed' 'Allowed' 'Blocked' 'Blocked' 'Blocked' 'Blocked'
 'Allowed' 'Blocked' 'Blocked' 'Blocked' 'Blocked' 'Allowed' 'Allowed']
